In [ ]:
# 导入必要的库
import os
import time
from fastkan import *
from fastkan import FastKAN
import random
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
import matplotlib.pyplot as plt
import matplotlib.patches as mpts
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, recall_score, cohen_kappa_score, accuracy_score
from sklearn.metrics import precision_score, precision_recall_curve, auc, recall_score, f1_score, accuracy_score, confusion_matrix
from sklearn.metrics import balanced_accuracy_score, f1_score, cohen_kappa_score
from sklearn.preprocessing import minmax_scale, StandardScaler
import pandas as pd
from scipy.io import loadmat
from tqdm.notebook import tqdm
from IPython import display
import h5py
import copy
import sys
import glob
import seaborn as sns
from datetime import datetime
import shap

%matplotlib inline

In [ ]:
## 超参数和实验设置配置单元格

# 设置随机种子，确保实验可重复性
RANDOM_SEED = 666

# 数据预处理参数
APPLY_PCA = True   # 是否应用PCA降维
NORM = True        # 是否对数据进行标准化/归一化处理

# 特征分组
DIFF_FEATURES = list(range(0, 15))     # 扩散特征 (1-15)
QTI_FEATURES = list(range(15, 225))    # QTI特征 (16-225)
CEST_FEATURES = list(range(225, 341))  # CEST特征 (226-341)

# 定义模型名称，用于结果保存和模型标识
MODEL_NAME = 'HierarchicalKAN_BrainVoxel'

# 大类定义 (初始设置，可能需要根据实际数据调整)
DEFAULT_NUM_BIG_CLASSES = 5  # 大类数量

# 指定数据集名称
DATASET = 'BrainVoxel'

# 训练参数
EPOCH = 50         # 总训练轮数
VAL_EPOCH = 1      # 每隔多少轮进行一次验证
LR = 0.001         # 学习率
WEIGHT_DECAY = 1e-6  # 权重衰减系数，用于L2正则化
BATCH_SIZE = 640    # 批处理大小，固定不变

# 计算设备选择
DEVICE = 0         # -1表示使用CPU，0表示使用第一块GPU(cuda:0)

# 数据参数
FEATURE_DIM = 341  # 输入特征总维度
NUM_CLASS = 102    # 细分类别数量
FIXED_GRID = 10    # 固定网格大小，不进行网格扩展

# 专家模型参数
DIFF_HIDDEN_DIM = 64    # 扩散专家隐藏层维度
QTI_HIDDEN_DIM = 128    # QTI专家隐藏层维度
CEST_HIDDEN_DIM = 64    # CEST专家隐藏层维度

# PCA参数
DIFF_PCA_COMPONENTS = 10   # 扩散特征PCA组件数
QTI_PCA_COMPONENTS = 30    # QTI特征PCA组件数
CEST_PCA_COMPONENTS = 20   # CEST特征PCA组件数

# 模型检查点路径
CHECK_POINT = None  # 加载预训练模型的路径，None表示从头开始训练

# 结果保存路径
SAVE_PATH = f"./Results/{MODEL_NAME}/{DATASET}"
# 如果保存目录不存在，则创建该目录
if not os.path.isdir(SAVE_PATH):
    os.makedirs(SAVE_PATH)

In [ ]:
## 设置随机数种子，确保实验结果可复现

# 为Python的random模块设置随机种子
random.seed(RANDOM_SEED)

# 为PyTorch的CPU操作设置随机种子
torch.manual_seed(RANDOM_SEED)

# 为当前GPU设置随机种子
torch.cuda.manual_seed(RANDOM_SEED)

# 为所有可用GPU设置相同的随机种子
torch.cuda.manual_seed_all(RANDOM_SEED)

# 为NumPy库设置随机种子
np.random.seed(RANDOM_SEED)

# 禁用CuDNN的非确定性算法
torch.backends.cudnn.deterministic = True

# 禁用CuDNN的自动优化选择
torch.backends.cudnn.benchmark = False

In [ ]:
def determine_optimal_big_classes(train_data, train_labels, feature_groups, min_classes=3, max_classes=8):
    """
    确定最佳的大类别数量，并生成映射关系
    
    参数：
        train_data: 训练数据
        train_labels: 训练标签
        feature_groups: 预处理后的特征分组
        min_classes: 需要考虑的大类别的最小数量
        max_classes: 需要考虑的大类别的最大数量
    
    返回：
        optimal_num_classes: 最优的大类别数量
        fine_to_big: 细类别到大类别的映射
        big_to_fine: 大类别到细类别的映射
        big_class_names: 大类别的名称
    """
    print("正在确定最佳的大类别数量...")
    
    # 合并多个特征组以进行聚类
    combined_features = np.hstack([
        feature_groups['diffusion'],
        feature_groups['qti'],
        feature_groups['cest']
    ])
    
    # 计算每个细类别的平均特征向量
    unique_classes = np.unique(train_labels)
    class_feature_vectors = {}
    
    for cls in unique_classes:
        mask = train_labels == cls
        if np.sum(mask) > 0:  # 只考虑包含样本的类别
            class_feature_vectors[cls] = np.mean(combined_features[mask], axis=0)
    
    # 转换为矩阵用于聚类
    class_ids = list(class_feature_vectors.keys())
    feature_matrix = np.array([class_feature_vectors[cls] for cls in class_ids])
    
    # 记录不同聚类数目的表现
    clustering_scores = []
    clustering_results = []
    
    from sklearn.cluster import AgglomerativeClustering
    from sklearn.metrics import silhouette_score, calinski_harabasz_score
    
    # 尝试不同数量的聚类
    for n_clusters in range(min_classes, max_classes + 1):
        # 执行层次聚类
        clustering = AgglomerativeClustering(n_clusters=n_clusters)
        cluster_labels = clustering.fit_predict(feature_matrix)
        
        # 计算聚类质量指标（如果样本数量足够）
        if len(feature_matrix) > n_clusters:
            silhouette = silhouette_score(feature_matrix, cluster_labels)
            calinski = calinski_harabasz_score(feature_matrix, cluster_labels)
            
            # 计算综合评分（归一化）
            combined_score = silhouette + calinski / 1000  # 缩放 Calinski-Harabasz 评分
            
            print(f"  {n_clusters} 个类别: 轮廓系数={silhouette:.4f}, Calinski-Harabasz={calinski:.2f}, 综合评分={combined_score:.4f}")
            
            clustering_scores.append(combined_score)
            clustering_results.append((n_clusters, cluster_labels, combined_score))
        else:
            print(f"  {n_clusters} 个类别: 样本不足，无法计算评估指标")
    
    # 选择最佳类别数量
    if clustering_scores:
        best_idx = np.argmax(clustering_scores)
        optimal_num_classes, best_cluster_labels, _ = clustering_results[best_idx]
        print(f"最佳的大类别数量: {optimal_num_classes}")
    else:
        # 如果无法计算评分，则使用默认值
        optimal_num_classes = (min_classes + max_classes) // 2
        best_idx = optimal_num_classes - min_classes
        best_cluster_labels = clustering_results[best_idx][1]
        print(f"使用默认值 {optimal_num_classes} 个大类别")
    
    # 创建类别映射
    fine_to_big = {}
    big_to_fine = {i: [] for i in range(optimal_num_classes)}
    
    for i, cls in enumerate(class_ids):
        cluster_id = best_cluster_labels[i]
        fine_to_big[cls] = cluster_id
        big_to_fine[cluster_id].append(cls)
    
    # 依据解剖学区域定义大类别名称
    brain_regions = {
        0: "白质 (WM)",
        1: "灰质 (GM)",
        2: "脑脊液 (CSF)",
        3: "深部核团 (DN)",
        4: "皮质下结构 (SS)",
        5: "小脑 (CB)",
        6: "脑干 (BS)",
        7: "其他区域 (Other)"
    }
    
    # 生成大类别名称，若超出定义的范围则使用默认名称
    big_class_names = []
    for i in range(optimal_num_classes):
        if i < len(brain_regions):
            big_class_names.append(brain_regions[i])
        else:
            big_class_names.append(f"区域 {i}")
    
    # 打印类别映射结果
    print("\n大类别映射:")
    for i in range(optimal_num_classes):
        print(f"  大类别 {i} ({big_class_names[i]}): {len(big_to_fine[i])} 个细类别")
        print(f"    细类别: {big_to_fine[i]}")
    
    # 可视化聚类结果
    visualize_big_class_clustering(feature_matrix, best_cluster_labels, class_ids, big_class_names)
    
    return optimal_num_classes, fine_to_big, big_to_fine, big_class_names

def visualize_big_class_clustering(feature_matrix, cluster_labels, class_ids, big_class_names):
    """
    可视化大类别（Big Class）的聚类结果
    
    参数：
        feature_matrix: 用于聚类的特征矩阵
        cluster_labels: 聚类标签（类别分配结果）
        class_ids: 原始类别 ID
        big_class_names: 大类别名称
    """
    # 使用 t-SNE 将高维数据降维到 2D 进行可视化
    from sklearn.manifold import TSNE
    
    # 进行降维
    tsne = TSNE(n_components=2, random_state=42)
    feature_matrix_2d = tsne.fit_transform(feature_matrix)
    
    # 创建绘图窗口
    plt.figure(figsize=(12, 10))
    
    # 获取所有唯一的大类别，并分配不同的颜色
    unique_clusters = np.unique(cluster_labels)
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_clusters)))
    
    # 绘制每个大类别
    for i, cluster in enumerate(unique_clusters):
        mask = cluster_labels == cluster
        plt.scatter(
            feature_matrix_2d[mask, 0], 
            feature_matrix_2d[mask, 1], 
            c=[colors[i]], 
            label=big_class_names[cluster],
            alpha=0.7
        )
    
    # 在图上标注每个类别的 ID
    for i, (x, y) in enumerate(feature_matrix_2d):
        plt.annotate(str(class_ids[i]), (x, y), fontsize=8)
    
    plt.title('大类别聚类可视化')
    plt.xlabel('t-SNE 组件 1')
    plt.ylabel('t-SNE 组件 2')
    plt.legend()
    plt.grid(True)
    
    # 保存可视化结果
    plt.savefig(os.path.join(SAVE_PATH, 'big_class_clustering.png'))
    plt.show()


In [ ]:
# 数据预处理和特征分组函数
def analyze_pca_variance(X, feature_group_name, max_components=None, plot=True, save_path=None):
    """
    分析PCA的方差解释率，找到合适的降维维度
    
    参数:
        X (ndarray): 输入数据
        feature_group_name (str): 特征组名称
        max_components (int): 最大考虑的主成分数，None表示使用特征维度
        plot (bool): 是否绘制解释方差曲线
        save_path (str): 保存图像的路径，None表示不保存
        
    返回:
        optimal_n_components: 建议的主成分数量
        pca_model: 训练好的PCA模型
    """
    # 确定最大主成分数
    if max_components is None:
        max_components = min(X.shape[0], X.shape[1])
    else:
        max_components = min(max_components, X.shape[0], X.shape[1])
    
    # 计算所有可能的主成分
    pca = PCA(n_components=max_components)
    pca.fit(X)
    
    # 计算累积解释方差
    explained_variance_ratio = pca.explained_variance_ratio_
    cumulative_variance_ratio = np.cumsum(explained_variance_ratio)
    
    # 寻找方差解释率达到95%的拐点
    threshold = 0.95
    optimal_n_components = np.argmax(cumulative_variance_ratio >= threshold) + 1
    
    # 寻找拐点（斜率变化最大的点）
    gradient = np.gradient(explained_variance_ratio)
    gradient_of_gradient = np.gradient(gradient)
    elbow_index = np.argmax(np.abs(gradient_of_gradient))
    elbow_n_components = elbow_index + 1
    
    if plot:
        plt.figure(figsize=(12, 6))
    
        # Plot Explained Variance Ratio
        plt.subplot(1, 2, 1)
        plt.plot(range(1, len(explained_variance_ratio) + 1), 
                 explained_variance_ratio, 'bo-', markersize=4)
        plt.axvline(x=elbow_n_components, color='r', linestyle='--', 
                    label=f'Elbow Point: {elbow_n_components} Components')
        plt.xlabel('Number of Principal Components')
        plt.ylabel('Explained Variance Ratio')
        plt.title(f'{feature_group_name} Features: Explained Variance Ratio')
        plt.grid(True)
        plt.legend()
    
        # Plot Cumulative Explained Variance
        plt.subplot(1, 2, 2)
        plt.plot(range(1, len(cumulative_variance_ratio) + 1), 
                 cumulative_variance_ratio, 'ro-', markersize=4)
        plt.axhline(y=threshold, color='g', linestyle='--', 
                    label=f'{threshold*100}% Variance')
        plt.axvline(x=optimal_n_components, color='b', linestyle='--', 
                    label=f'Threshold Components: {optimal_n_components}')
        plt.xlabel('Number of Principal Components')
        plt.ylabel('Cumulative Explained Variance Ratio')
        plt.title(f'{feature_group_name} Features: Cumulative Variance')
        plt.grid(True)
        plt.legend()
    
        plt.tight_layout()
    
        if save_path:
            plt.savefig(save_path)
        plt.show()
    
    print(f"{feature_group_name} 特征组:")
    print(f"方差拐点对应的主成分数量: {elbow_n_components}")
    print(f"达到{threshold*100}%方差解释率需要的主成分数量: {optimal_n_components}")
    print(f"前{optimal_n_components}个主成分解释了总方差的{cumulative_variance_ratio[optimal_n_components-1]*100:.2f}%")
    
    # 创建新的PCA模型，使用最佳组件数
    pca_model = PCA(n_components=optimal_n_components)
    pca_model.fit(X)
    
    return optimal_n_components, pca_model

def preprocess_feature_group(data, group_name, feature_indices, apply_pca=True, n_components=None, normalize=True):
    """
    对特定特征组进行预处理
    
    参数:
        data (ndarray): 完整数据集
        group_name (str): 特征组名称
        feature_indices (list): 特征组对应的索引
        apply_pca (bool): 是否应用PCA
        n_components (int): PCA组件数，None表示自动选择
        normalize (bool): 是否进行标准化
        
    返回:
        processed_data: 处理后的数据
        pca_model: PCA模型（如果使用PCA）
    """
    # 提取特征组数据
    group_data = data[:, feature_indices]
    
    print(f"\n处理 {group_name} 特征组 (维度: {group_data.shape[1]})...")
    
    # 标准化
    if normalize:
        print(f"对 {group_name} 特征进行标准化...")
        scaler = StandardScaler()
        group_data = scaler.fit_transform(group_data)
    
    # 应用PCA
    pca_model = None
    if apply_pca:
        print(f"对 {group_name} 特征应用PCA...")
        if n_components is None or n_components <= 0:
            # 自动确定PCA组件数
            save_path = os.path.join(SAVE_PATH, f"{group_name}_pca_variance.png")
            _, pca_model = analyze_pca_variance(group_data, group_name, plot=True, save_path=save_path)
            processed_data = pca_model.transform(group_data)
            print(f"自动选择的PCA组件数: {pca_model.n_components_}")
        else:
            # 使用指定的PCA组件数
            pca_model = PCA(n_components=n_components)
            processed_data = pca_model.fit_transform(group_data)
            print(f"使用指定的PCA组件数: {n_components}")
            
            # 计算解释方差比例
            var_ratio = np.sum(pca_model.explained_variance_ratio_)
            print(f"PCA解释的方差比例: {var_ratio:.4f}")
    else:
        processed_data = group_data
        print(f"不使用PCA降维，保持原始维度: {group_data.shape[1]}")
    
    return processed_data, pca_model

def preprocess_all_feature_groups(data, apply_pca=True, normalize=True):
    """
    对所有特征组进行预处理
    
    参数:
        data (ndarray): 完整数据集
        apply_pca (bool): 是否应用PCA
        normalize (bool): 是否进行标准化
        
    返回:
        processed_groups: 包含处理后所有特征组的字典
        pca_models: 包含所有PCA模型的字典
    """
    processed_groups = {}
    pca_models = {}
    
    # 处理扩散特征组
    diff_data, diff_pca = preprocess_feature_group(
        data, 'diffusion', DIFF_FEATURES, 
        apply_pca=apply_pca, 
        n_components=DIFF_PCA_COMPONENTS, 
        normalize=normalize
    )
    processed_groups['diffusion'] = diff_data
    pca_models['diffusion'] = diff_pca
    
    # 处理QTI特征组
    qti_data, qti_pca = preprocess_feature_group(
        data, 'qti', QTI_FEATURES, 
        apply_pca=apply_pca, 
        n_components=QTI_PCA_COMPONENTS, 
        normalize=normalize
    )
    processed_groups['qti'] = qti_data
    pca_models['qti'] = qti_pca
    
    # 处理CEST特征组
    cest_data, cest_pca = preprocess_feature_group(
        data, 'cest', CEST_FEATURES, 
        apply_pca=apply_pca, 
        n_components=CEST_PCA_COMPONENTS, 
        normalize=normalize
    )
    processed_groups['cest'] = cest_data
    pca_models['cest'] = cest_pca
    
    # 输出处理后的特征组信息
    for group_name, group_data in processed_groups.items():
        print(f"{group_name} 特征组处理后维度: {group_data.shape}")
    
    return processed_groups, pca_models

In [ ]:
# 特征重要性分析函数
def analyze_feature_importance(feature_groups, big_class_labels, save_path=None):
    """
    分析不同特征组对大类分类的重要性
    
    参数:
        feature_groups (dict): 包含处理后特征组的字典
        big_class_labels (array): 大类标签
        save_path (str): 保存结果的路径
    
    返回:
        importance_dict: 包含各特征组重要性分数的字典
    """
    importance_dict = {}
    
    plt.figure(figsize=(15, 10))
    
    # 分析每个特征组
    for i, (group_name, group_data) in enumerate(feature_groups.items()):
        print(f"\n分析 {group_name} 特征组的重要性...")
        
        # 使用F统计量评估特征重要性
        selector = SelectKBest(f_classif, k='all')
        selector.fit(group_data, big_class_labels)
        
        # 保存特征重要性分数
        scores = selector.scores_
        importance_dict[group_name] = {
            'mean_score': np.mean(scores),
            'max_score': np.max(scores),
            'min_score': np.min(scores),
            'scores': scores
        }
        
        # 绘制特征重要性分布
        plt.subplot(2, 2, i+1)
        plt.bar(range(len(scores)), scores)
        plt.title(f'{group_name} Feature Importance')
        plt.xlabel('Feature Index')
        plt.ylabel('F-score')
        plt.grid(True)
        
        print(f"{group_name} 特征组平均重要性分数: {np.mean(scores):.4f}")
        print(f"{group_name} 特征组最大重要性分数: {np.max(scores):.4f}")
        print(f"{group_name} 特征组最小重要性分数: {np.min(scores):.4f}")
    
    # 比较各特征组的平均重要性
    plt.subplot(2, 2, 4)
    group_names = list(importance_dict.keys())
    mean_scores = [importance_dict[name]['mean_score'] for name in group_names]
    plt.bar(group_names, mean_scores)
    plt.title('Average Feature Importance by Group')
    plt.ylabel('Mean F-score')
    plt.grid(True)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
    plt.show()
    
    return importance_dict

def visualize_feature_distributions(feature_groups, big_class_labels, save_path=None):
    """
    可视化不同大类在各特征组上的分布
    
    参数:
        feature_groups (dict): 包含处理后特征组的字典
        big_class_labels (array): 大类标签
        save_path (str): 保存结果的路径
    """
    # 获取唯一的大类标签
    unique_classes = np.unique(big_class_labels)
    num_classes = len(unique_classes)
    
    # 为每个特征组创建可视化
    for group_name, group_data in feature_groups.items():
        print(f"\n可视化 {group_name} 特征组的类别分布...")
        
        # 选择前两个主成分进行可视化
        if group_data.shape[1] > 2:
            pca = PCA(n_components=2)
            data_2d = pca.fit_transform(group_data)
            title_suffix = " (PCA 2D projection)"
        else:
            data_2d = group_data
            title_suffix = ""
        
        plt.figure(figsize=(12, 10))
        
        # 为每个类别绘制散点图
        for class_id in unique_classes:
            mask = big_class_labels == class_id
            plt.scatter(data_2d[mask, 0], data_2d[mask, 1], alpha=0.6, label=f'Class {class_id}')
        
        plt.title(f'{group_name} Feature Distribution by Class{title_suffix}')
        plt.xlabel('Component 1')
        plt.ylabel('Component 2')
        plt.legend()
        plt.grid(True)
        
        if save_path:
            plt.savefig(os.path.join(save_path, f"{group_name}_class_distribution.png"))
        plt.show()

In [ ]:
#  大类标签定义函数
def define_big_classes():
    """
    定义大类标签映射关系
    这个函数根据生物学和解剖学知识，将102个细分类别映射到几个大类
    
    返回:
        fine_to_big: 细分类别到大类的映射字典
        big_to_fine: 大类到细分类别的映射字典
        big_class_names: 大类名称列表
    """
    # 定义大类名称
    big_class_names = [
        "白质区域 (White Matter)",  # 0
        "灰质区域 (Gray Matter)",   # 1
        "脑脊液 (CSF)",            # 2
        "深层核团 (Deep Nuclei)",   # 3
        "其他区域 (Other)"          # 4
    ]
    
    # 初始化映射字典
    fine_to_big = {}
    big_to_fine = {i: [] for i in range(len(big_class_names))}
    
    # 白质区域 (类别0): 典型白质结构
    white_matter_classes = [0, 3, 4, 6, 28, 33, 34, 35, 38, 53, 56]
    for cls in white_matter_classes:
        fine_to_big[cls] = 0
        big_to_fine[0].append(cls)
    
    # 灰质区域 (类别1): 典型灰质结构
    gray_matter_classes = [10, 16, 37, 39, 41, 46, 57, 60, 67, 68, 71, 72, 73, 87, 90, 94, 95]
    for cls in gray_matter_classes:
        fine_to_big[cls] = 1
        big_to_fine[1].append(cls)
    
    # 脑脊液 (类别2): CSF相关区域
    csf_classes = [5, 7, 11, 12, 47, 74, 75]
    for cls in csf_classes:
        fine_to_big[cls] = 2
        big_to_fine[2].append(cls)
    
    # 深层核团 (类别3): 基底核、丘脑等
    deep_nuclei_classes = [1, 2, 31, 40, 44, 45, 52, 54, 62, 63, 69, 70, 76, 80, 81, 85, 86]
    for cls in deep_nuclei_classes:
        fine_to_big[cls] = 3
        big_to_fine[3].append(cls)
    
    # 其他区域 (类别4): 包括所有其他类别
    other_classes = [cls for cls in range(NUM_CLASS) if cls not in 
                    white_matter_classes + gray_matter_classes + 
                    csf_classes + deep_nuclei_classes]
    for cls in other_classes:
        fine_to_big[cls] = 4
        big_to_fine[4].append(cls)
    
    # 打印大类统计信息
    print("大类划分统计:")
    for i, name in enumerate(big_class_names):
        print(f"大类 {i} - {name}: {len(big_to_fine[i])} 个细分类别")
        print(f"    包含的细分类别: {big_to_fine[i]}")
    
    return fine_to_big, big_to_fine, big_class_names

def map_to_big_classes(fine_labels, fine_to_big):
    """
    将细分类别标签映射为大类标签
    
    参数:
        fine_labels (array): 细分类别标签
        fine_to_big (dict): 细分类别到大类的映射字典
    
    返回:
        big_labels: 大类标签
    """
    big_labels = np.array([fine_to_big.get(label, 4) for label in fine_labels])
    return big_labels

In [ ]:
# 数据集类定义
class BrainVoxelDataset(Dataset):
    """脑体素数据集类"""
    def __init__(self, feature_groups, labels, is_inference=False):
        """
        初始化数据集
        
        参数:
            feature_groups (dict): 包含处理后特征组的字典
            labels: 标签数据
            is_inference: 是否为推理模式（不返回标签）
        """
        super(BrainVoxelDataset, self).__init__()
        self.feature_groups = feature_groups
        self.labels = labels
        self.is_inference = is_inference
        
        # 获取所有样本数量(所有特征组应有相同的样本数)
        group_name = list(feature_groups.keys())[0]
        self.num_samples = feature_groups[group_name].shape[0]
        
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        # 创建一个包含所有特征组的字典
        features = {}
        for group_name, group_data in self.feature_groups.items():
            features[group_name] = torch.FloatTensor(group_data[idx])
        
        if self.is_inference:
            return features
        else:
            label = self.labels[idx]
            label = torch.LongTensor([int(label)])[0]  # 确保标签是整数
            return features, label

In [ ]:
# 专家模型定义
class ExpertKAN(nn.Module):
    """专家KAN模型，用于处理特定特征组"""
    def __init__(self, input_dim, hidden_dim, num_classes, grid_size=10):
        """
        初始化专家模型
        
        参数:
            input_dim: 输入特征维度
            hidden_dim: 隐藏层维度
            num_classes: 输出类别数量
            grid_size: 网格大小
        """
        super(ExpertKAN, self).__init__()
        self.kan = FastKAN(
            layers_hidden=[input_dim, hidden_dim, num_classes],
            num_grids=grid_size
        )
    
    def forward(self, x):
        """前向传播"""
        return self.kan(x)

class EnsembleExpertSystem(nn.Module):
    """集成多个专家模型的系统"""
    def __init__(self, expert_models, expert_weights=None):
        """
        初始化集成专家系统
        
        参数:
            expert_models (dict): 包含专家模型的字典
            expert_weights (dict): 专家权重字典，默认为None（平均权重）
        """
        super(EnsembleExpertSystem, self).__init__()
        self.expert_models = expert_models
        
        # 如果未提供权重，则使用平均权重
        if expert_weights is None:
            num_experts = len(expert_models)
            self.expert_weights = {name: 1.0/num_experts for name in expert_models.keys()}
        else:
            self.expert_weights = expert_weights
    
    def forward(self, features_dict):
        """
        前向传播
        
        参数:
            features_dict (dict): 包含各特征组数据的字典
        
        返回:
            ensemble_output: 集成输出
        """
        # 收集所有专家的输出
        expert_outputs = {}
        for name, model in self.expert_models.items():
            expert_outputs[name] = model(features_dict[name])
        
        # 加权融合专家输出
        ensemble_output = None
        for name, output in expert_outputs.items():
            if ensemble_output is None:
                ensemble_output = self.expert_weights[name] * output
            else:
                ensemble_output += self.expert_weights[name] * output
        
        return ensemble_output, expert_outputs

In [ ]:
# 训练函数
def train_expert_model(expert_name, model, train_loader, val_loader, criterion, optimizer, device, 
                      num_epochs=50, val_epoch=1, save_path="./Results"):
    """
    训练专家模型
    
    参数:
        expert_name (str): 专家模型名称
        model: 模型
        train_loader: 训练数据加载器
        val_loader: 验证数据加载器
        criterion: 损失函数
        optimizer: 优化器
        device: 计算设备
        num_epochs: 训练轮数
        val_epoch: 验证频率
        save_path: 结果保存路径
    
    返回:
        训练结果字典
    """
    # 创建专家模型的保存目录
    expert_save_path = os.path.join(save_path, expert_name)
    os.makedirs(expert_save_path, exist_ok=True)
    
    # 初始化统计变量
    loss_list = []
    acc_list = []
    val_acc_list = []
    val_epoch_list = []
    
    # 保存起始时间
    train_st = time.time()
    
    # 计算批次数量和样本数量
    batch_num = len(train_loader)
    train_num = len(train_loader.dataset)
    
    try:
        # 训练循环
        for e in range(num_epochs):
            # 设置模型为训练模式
            model.train()
            avg_loss = 0.0
            
            # 收集训练数据的预测结果
            train_preds = []
            train_targets = []
            
            # 批次循环
            for batch_idx, (features, target) in enumerate(train_loader):
                # 获取当前专家的特征
                data = features[expert_name].to(device)
                target = target.to(device)
                
                # 前向传播
                optimizer.zero_grad()
                out = model(data)
                loss = criterion(out, target)
                
                # 反向传播
                loss.backward()
                optimizer.step()
                
                # 累计损失
                avg_loss += loss.item()
                
                # 收集预测结果用于指标计算
                _, pred = torch.max(out, dim=1)
                train_preds.extend(pred.cpu().numpy())
                train_targets.extend(target.cpu().numpy())
                
                # 显示进度条
                if (batch_idx + 1) % 10 == 0:
                    print(f"\r{expert_name} Epoch: {e+1}/{num_epochs} [{batch_idx+1}/{batch_num}] Loss: {loss.item():.6f}", end="")
            
            # 计算训练集指标
            train_acc = accuracy_score(train_targets, train_preds)
            
            # 保存训练集指标
            loss_list.append(avg_loss / train_num)
            acc_list.append(train_acc)
            
            # 显示训练指标
            print(f"\n{expert_name} epoch {e+1}/{num_epochs} loss:{loss_list[-1]:.6f} acc:{train_acc:.4f}")
            
            # 验证阶段
            if (e+1) % val_epoch == 0 or (e+1) == num_epochs:
                model.eval()
                
                # 收集验证数据的预测结果
                val_preds = []
                val_targets = []
                
                with torch.no_grad():
                    for features, target in val_loader:
                        # 获取当前专家的特征
                        data = features[expert_name].to(device)
                        target = target.to(device)
                        
                        out = model(data)
                        _, pred = torch.max(out, dim=1)
                        
                        val_preds.extend(pred.cpu().numpy())
                        val_targets.extend(target.cpu().numpy())
                
                # 计算验证集指标
                val_acc = accuracy_score(val_targets, val_preds)
                
                # 保存验证集指标
                val_acc_list.append(val_acc)
                val_epoch_list.append(e)
                
                # 显示验证集指标
                print(f"{expert_name} [验证集] acc:{val_acc:.4f}")
                
                # 计算混淆矩阵
                conf_matrix = confusion_matrix(val_targets, val_preds)
                
                # 保存当前模型
                save_name = os.path.join(expert_save_path, f"epoch_{e+1}_acc_{val_acc:.4f}.pth")
                save_dict = {
                    'state_dict': model.state_dict(), 
                    'epoch': e+1, 
                    'optimizer': optimizer.state_dict(),
                    'loss_list': loss_list, 
                    'acc_list': acc_list, 
                    'val_acc_list': val_acc_list, 
                    'val_epoch_list': val_epoch_list
                }
                torch.save(save_dict, save_name)
                
                # 可视化混淆矩阵
                if (e+1) == num_epochs:
                    plt.figure(figsize=(10, 8))
                    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
                    plt.title(f'{expert_name} Confusion Matrix (Epoch {e+1})')
                    plt.xlabel('Predicted')
                    plt.ylabel('True')
                    plt.savefig(os.path.join(expert_save_path, f'confusion_matrix_epoch_{e+1}.png'))
                    plt.close()
                
    except Exception as exc:
        print(f"Error during training {expert_name}: {exc}")
        import traceback
        traceback.print_exc()
        
    finally:
        print(f'{expert_name} 训练停止于epoch {e+1}')
    
    # 计算总训练时间
    train_time = time.time() - train_st
    print(f"{expert_name} 训练时间: {train_time:.2f}秒")
    
    # 绘制训练曲线
    plt.figure(figsize=(15, 5))
    
    # 损失曲线
    plt.subplot(1, 3, 1)
    plt.plot(range(len(loss_list)), loss_list)
    plt.title(f'{expert_name} Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    
    # 准确率曲线
    plt.subplot(1, 3, 2)
    plt.plot(range(len(acc_list)), acc_list, label='Train Acc')
    plt.plot(val_epoch_list, val_acc_list, label='Val Acc')
    plt.title(f'{expert_name} Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(expert_save_path, 'training_curves.png'))
    plt.close()
    
    # 返回训练结果
    return {
        'model': model,
        'loss_list': loss_list,
        'acc_list': acc_list,
        'val_acc_list': val_acc_list,
        'val_epoch_list': val_epoch_list,
        'train_time': train_time,
        'best_val_acc': max(val_acc_list) if val_acc_list else 0,
        'best_epoch': val_epoch_list[np.argmax(val_acc_list)] if val_acc_list else 0
    }

In [ ]:
# Ensemble Training Function 集成训练函数
def train_ensemble_model(ensemble_model, train_loader, val_loader, criterion, optimizer, device, 
                        num_epochs=50, val_epoch=1, save_path="./Results"):
    """
    训练集成模型
    
    参数:
        ensemble_model: 集成模型
        train_loader: 训练数据加载器
        val_loader: 验证数据加载器
        criterion: 损失函数
        optimizer: 优化器
        device: 计算设备
        num_epochs: 训练轮数
        val_epoch: 验证频率
        save_path: 结果保存路径
    
    返回:
        训练结果字典
    """
    # 创建集成模型的保存目录
    ensemble_save_path = os.path.join(save_path, "ensemble")
    os.makedirs(ensemble_save_path, exist_ok=True)
    
    # 初始化统计变量
    loss_list = []
    acc_list = []
    val_acc_list = []
    val_epoch_list = []
    
    # 保存起始时间
    train_st = time.time()
    
    # 计算批次数量和样本数量
    batch_num = len(train_loader)
    train_num = len(train_loader.dataset)
    
    try:
        # 训练循环
        for e in range(num_epochs):
            # 设置模型为训练模式
            ensemble_model.train()
            avg_loss = 0.0
            
            # 收集训练数据的预测结果
            train_preds = []
            train_targets = []
            
            # 批次循环
            for batch_idx, (features, target) in enumerate(train_loader):
                # 将特征移动到设备
                features_on_device = {k: v.to(device) for k, v in features.items()}
                target = target.to(device)
                
                # 前向传播
                optimizer.zero_grad()
                out, _ = ensemble_model(features_on_device)
                loss = criterion(out, target)
                
                # 反向传播
                loss.backward()
                optimizer.step()
                
                # 累计损失
                avg_loss += loss.item()
                
                # 收集预测结果用于指标计算
                _, pred = torch.max(out, dim=1)
                train_preds.extend(pred.cpu().numpy())
                train_targets.extend(target.cpu().numpy())
                
                # 显示进度条
                if (batch_idx + 1) % 10 == 0:
                    print(f"\rEnsemble Epoch: {e+1}/{num_epochs} [{batch_idx+1}/{batch_num}] Loss: {loss.item():.6f}", end="")
            
            # 计算训练集指标
            train_acc = accuracy_score(train_targets, train_preds)
            
            # 保存训练集指标
            loss_list.append(avg_loss / train_num)
            acc_list.append(train_acc)
            
            # 显示训练指标
            print(f"\nEnsemble epoch {e+1}/{num_epochs} loss:{loss_list[-1]:.6f} acc:{train_acc:.4f}")
            
            # 验证阶段
            if (e+1) % val_epoch == 0 or (e+1) == num_epochs:
                ensemble_model.eval()
                
                # 收集验证数据的预测结果
                val_preds = []
                val_targets = []
                expert_val_preds = {name: [] for name in ensemble_model.expert_models.keys()}
                
                with torch.no_grad():
                    for features, target in val_loader:
                        # 将特征移动到设备
                        features_on_device = {k: v.to(device) for k, v in features.items()}
                        target = target.to(device)
                        
                        out, expert_outputs = ensemble_model(features_on_device)
                        _, pred = torch.max(out, dim=1)
                        
                        val_preds.extend(pred.cpu().numpy())
                        val_targets.extend(target.cpu().numpy())
                        
                        # 记录每个专家的预测
                        for name, expert_out in expert_outputs.items():
                            _, expert_pred = torch.max(expert_out, dim=1)
                            expert_val_preds[name].extend(expert_pred.cpu().numpy())
                
                # 计算验证集指标
                val_acc = accuracy_score(val_targets, val_preds)
                
                # 计算每个专家的准确率
                expert_accuracies = {}
                for name, preds in expert_val_preds.items():
                    expert_acc = accuracy_score(val_targets, preds)
                    expert_accuracies[name] = expert_acc
                
                # 保存验证集指标
                val_acc_list.append(val_acc)
                val_epoch_list.append(e)
                
                # 显示验证集指标
                print(f"Ensemble [验证集] acc:{val_acc:.4f}")
                print("各专家模型验证集准确率:")
                for name, acc in expert_accuracies.items():
                    print(f"  {name}: {acc:.4f}")
                
                # 计算混淆矩阵
                conf_matrix = confusion_matrix(val_targets, val_preds)
                
                # 保存当前模型
                save_name = os.path.join(ensemble_save_path, f"epoch_{e+1}_acc_{val_acc:.4f}.pth")
                save_dict = {
                    'state_dict': ensemble_model.state_dict(), 
                    'expert_weights': ensemble_model.expert_weights,
                    'epoch': e+1, 
                    'optimizer': optimizer.state_dict(),
                    'loss_list': loss_list, 
                    'acc_list': acc_list, 
                    'val_acc_list': val_acc_list, 
                    'val_epoch_list': val_epoch_list,
                    'expert_accuracies': expert_accuracies
                }
                torch.save(save_dict, save_name)
                
                # 可视化混淆矩阵
                if (e+1) == num_epochs:
                    plt.figure(figsize=(10, 8))
                    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
                    plt.title(f'Ensemble Confusion Matrix (Epoch {e+1})')
                    plt.xlabel('Predicted')
                    plt.ylabel('True')
                    plt.savefig(os.path.join(ensemble_save_path, f'confusion_matrix_epoch_{e+1}.png'))
                    plt.close()
                
    except Exception as exc:
        print(f"Error during ensemble training: {exc}")
        import traceback
        traceback.print_exc()
        
    finally:
        print(f'Ensemble 训练停止于epoch {e+1}')
    
    # 计算总训练时间
    train_time = time.time() - train_st
    print(f"Ensemble 训练时间: {train_time:.2f}秒")
    
    # 绘制训练曲线
    plt.figure(figsize=(15, 5))
    
    # 损失曲线
    plt.subplot(1, 3, 1)
    plt.plot(range(len(loss_list)), loss_list)
    plt.title('Ensemble Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    
    # 准确率曲线
    plt.subplot(1, 3, 2)
    plt.plot(range(len(acc_list)), acc_list, label='Train Acc')
    plt.plot(val_epoch_list, val_acc_list, label='Val Acc')
    plt.title('Ensemble Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(ensemble_save_path, 'training_curves.png'))
    plt.close()
    
    # 返回训练结果
    return {
        'model': ensemble_model,
        'loss_list': loss_list,
        'acc_list': acc_list,
        'val_acc_list': val_acc_list,
        'val_epoch_list': val_epoch_list,
        'train_time': train_time,
        'best_val_acc': max(val_acc_list) if val_acc_list else 0,
        'best_epoch': val_epoch_list[np.argmax(val_acc_list)] if val_acc_list else 0
    }

In [ ]:
# Model Evaluation Functions 模型评估函数
def evaluate_model(model, data_loader, device, is_ensemble=False):
    """
    评估模型性能
    
    参数:
        model: 模型
        data_loader: 数据加载器
        device: 计算设备
        is_ensemble: 是否为集成模型
    
    返回:
        evaluation_results: 评估结果字典
    """
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for data in data_loader:
            if is_ensemble:
                features, target = data
                # 将特征移动到设备
                features_on_device = {k: v.to(device) for k, v in features.items()}
                target = target.to(device)
                
                output, _ = model(features_on_device)
                _, preds = torch.max(output, 1)
            else:
                # 单专家模型评估
                features, target = data
                # 获取特定特征
                feature_name = list(features.keys())[0]
                feature = features[feature_name].to(device)
                target = target.to(device)
                
                output = model(feature)
                _, preds = torch.max(output, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(target.cpu().numpy())
    
    # 计算评估指标
    accuracy = accuracy_score(all_targets, all_preds)
    balanced_acc = balanced_accuracy_score(all_targets, all_preds)
    f1_macro = f1_score(all_targets, all_preds, average='macro')
    f1_weighted = f1_score(all_targets, all_preds, average='weighted')
    
    # 计算每个类别的准确率
    conf_matrix = confusion_matrix(all_targets, all_preds)
    num_classes = conf_matrix.shape[0]  # 从混淆矩阵获取类别数量
    per_class_accuracy = np.zeros(num_classes)
    for i in range(num_classes):
        if np.sum(np.array(all_targets) == i) > 0:
            per_class_accuracy[i] = np.sum((np.array(all_targets) == i) & (np.array(all_preds) == i)) / np.sum(np.array(all_targets) == i)
    
    # 生成分类报告
    report = classification_report(all_targets, all_preds, digits=4)
    
    return {
        'accuracy': accuracy,
        'balanced_accuracy': balanced_acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'per_class_accuracy': per_class_accuracy,
        'confusion_matrix': conf_matrix,
        'classification_report': report,
        'predictions': all_preds,
        'targets': all_targets
    }

def visualize_evaluation_results(results, class_names, save_path=None):
    """
    可视化评估结果
    
    参数:
        results: 评估结果字典
        class_names: 类别名称列表
        save_path: 保存路径
    """
    # class_names 的长度应该等于 optimal_num_classes
    # 绘制混淆矩阵
    plt.figure(figsize=(12, 10))
    sns.heatmap(results['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
               xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    
    if save_path:
        plt.savefig(os.path.join(save_path, 'confusion_matrix.png'))
    plt.show()
    
    # 绘制每类准确率
    plt.figure(figsize=(12, 6))
    plt.bar(class_names, results['per_class_accuracy'])
    plt.title('Per-Class Accuracy')
    plt.xlabel('Class')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1)
    plt.xticks(rotation=45)
    plt.grid(axis='y')
    
    # 添加准确率数值
    for i, v in enumerate(results['per_class_accuracy']):
        plt.text(i, v + 0.02, f"{v:.2f}", ha='center')
    
    if save_path:
        plt.savefig(os.path.join(save_path, 'per_class_accuracy.png'))
    plt.show()
    
    # 打印主要指标
    print(f"整体准确率: {results['accuracy']:.4f}")
    print(f"平衡准确率: {results['balanced_accuracy']:.4f}")
    print(f"宏平均F1: {results['f1_macro']:.4f}")
    print(f"加权F1: {results['f1_weighted']:.4f}")
    print("\n分类报告:")
    print(results['classification_report'])

In [ ]:
def evaluate_on_all_datasets(ensemble_model, pca_models, fine_to_big, big_class_names, data_dirs, save_path):
    """
    对所有数据集（训练集、测试集、验证集、合并集）进行评估
    
    参数:
        ensemble_model: 集成模型
        pca_models: PCA模型字典
        fine_to_big: 细分类别到大类的映射
        big_class_names: 大类名称列表
        data_dirs: 数据目录字典
        save_path: 结果保存路径
        
    返回:
        all_results: 包含所有评估结果的字典
    """
    # 设置计算设备
    device = torch.device(f"cuda:{DEVICE}" if DEVICE>=0 and torch.cuda.is_available() else "cpu")
    
    # 创建结果存储目录
    results_dir = os.path.join(save_path, "evaluation_results")
    os.makedirs(results_dir, exist_ok=True)
    
    # 加载所有数据集
    print("\n加载所有数据集...")
    from load_multiclass_data_from_dirs import load_multiclass_data_from_dirs
    
    dataset_dict = load_multiclass_data_from_dirs(
        data_dirs, 
        apply_pca=False,
        n_components=0, 
        norm=False
    )
    
    # 加载合并数据集（如果提供）
    if 'merged_dir' in data_dirs:
        from load_merged_dataset import load_merged_dataset
        merged_data, merged_labels = load_merged_dataset(
            data_dirs['merged_dir'],
            apply_pca=False,
            pca_model=None,
            norm=False
        )
    else:
        # 如果没有提供合并目录，则合并训练、测试和验证集
        merged_data = np.vstack([dataset_dict['train_samples'], 
                                dataset_dict['test_samples'], 
                                dataset_dict['val_samples']])
        merged_labels = np.concatenate([dataset_dict['train_labels'], 
                                       dataset_dict['test_labels'], 
                                       dataset_dict['val_labels']])
    
    # 将细分类标签映射为大类标签
    train_big_labels = map_to_big_classes(dataset_dict['train_labels'], fine_to_big)
    test_big_labels = map_to_big_classes(dataset_dict['test_labels'], fine_to_big)
    val_big_labels = map_to_big_classes(dataset_dict['val_labels'], fine_to_big)
    merged_big_labels = map_to_big_classes(merged_labels, fine_to_big)
    
    # 处理所有数据集
    print("\n处理数据集...")
    train_groups = prepare_data_for_inference(dataset_dict['train_samples'], pca_models)
    test_groups = prepare_data_for_inference(dataset_dict['test_samples'], pca_models)
    val_groups = prepare_data_for_inference(dataset_dict['val_samples'], pca_models)
    merged_groups = prepare_data_for_inference(merged_data, pca_models)
    
    # 创建数据集和加载器
    train_dataset = BrainVoxelDataset(train_groups, train_big_labels)
    test_dataset = BrainVoxelDataset(test_groups, test_big_labels)
    val_dataset = BrainVoxelDataset(val_groups, val_big_labels)
    merged_dataset = BrainVoxelDataset(merged_groups, merged_big_labels)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    merged_loader = DataLoader(merged_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # 评估所有数据集
    print("\n评估训练集...")
    train_results = evaluate_model(ensemble_model, train_loader, device, is_ensemble=True)
    
    print("\n评估测试集...")
    test_results = evaluate_model(ensemble_model, test_loader, device, is_ensemble=True)
    
    print("\n评估验证集...")
    val_results = evaluate_model(ensemble_model, val_loader, device, is_ensemble=True)
    
    print("\n评估合并集...")
    merged_results = evaluate_model(ensemble_model, merged_loader, device, is_ensemble=True)
    
    # 保存结果
    all_results = {
        'train': train_results,
        'test': test_results,
        'val': val_results,
        'merged': merged_results
    }
    
    # 创建比较可视化
    plt.figure(figsize=(15, 10))
    
    # 整体准确率比较 - 使用英文标签
    plt.subplot(2, 2, 1)
    datasets = ['Training Set', 'Test Set', 'Validation Set', 'Merged Set']
    accuracies = [train_results['accuracy'], test_results['accuracy'], 
                 val_results['accuracy'], merged_results['accuracy']]
    plt.bar(datasets, accuracies, color=['blue', 'green', 'orange', 'red'])
    plt.title('Overall Accuracy Comparison')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1.0)
    plt.grid(axis='y')
    
    for i, v in enumerate(accuracies):
        plt.text(i, v + 0.02, f"{v:.4f}", ha='center')
    
    # 各类别在不同数据集上的准确率 - 使用英文标签
    plt.subplot(2, 2, 2)
    x = np.arange(len(big_class_names))
    width = 0.2
    
    # 准备英文类别名称（如果原始名称是中文）
    english_class_names = []
    for name in big_class_names:
        # 提取括号中的英文部分，如果存在
        if '(' in name and ')' in name:
            eng_name = name[name.find('(')+1:name.find(')')]
            english_class_names.append(eng_name)
        else:
            english_class_names.append(name)
    
    plt.bar(x - 1.5*width, train_results['per_class_accuracy'], width, label='Training Set')
    plt.bar(x - 0.5*width, test_results['per_class_accuracy'], width, label='Test Set')
    plt.bar(x + 0.5*width, val_results['per_class_accuracy'], width, label='Validation Set')
    plt.bar(x + 1.5*width, merged_results['per_class_accuracy'], width, label='Merged Set')
    
    plt.xlabel('Class')
    plt.ylabel('Accuracy')
    plt.title('Per-Class Accuracy Across Datasets')
    plt.xticks(x, english_class_names, rotation=45)
    plt.legend()
    plt.grid(axis='y')
    
    # F1得分比较 - 使用英文标签
    plt.subplot(2, 2, 3)
    datasets = ['Training Set', 'Test Set', 'Validation Set', 'Merged Set']
    f1_macro = [train_results['f1_macro'], test_results['f1_macro'], 
               val_results['f1_macro'], merged_results['f1_macro']]
    f1_weighted = [train_results['f1_weighted'], test_results['f1_weighted'], 
                  val_results['f1_weighted'], merged_results['f1_weighted']]
    
    x = np.arange(len(datasets))
    width = 0.35
    
    plt.bar(x - width/2, f1_macro, width, label='Macro F1')
    plt.bar(x + width/2, f1_weighted, width, label='Weighted F1')
    
    plt.xlabel('Dataset')
    plt.ylabel('F1 Score')
    plt.title('F1 Score Comparison')
    plt.xticks(x, datasets)
    plt.legend()
    plt.grid(axis='y')
    
    # 平衡准确率比较 - 使用英文标签
    plt.subplot(2, 2, 4)
    balanced_acc = [train_results['balanced_accuracy'], test_results['balanced_accuracy'], 
                   val_results['balanced_accuracy'], merged_results['balanced_accuracy']]
    
    plt.bar(datasets, balanced_acc, color=['blue', 'green', 'orange', 'red'])
    plt.title('Balanced Accuracy Comparison')
    plt.ylabel('Balanced Accuracy')
    plt.ylim(0, 1.0)
    plt.grid(axis='y')
    
    for i, v in enumerate(balanced_acc):
        plt.text(i, v + 0.02, f"{v:.4f}", ha='center')
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'all_datasets_comparison.png'))
    plt.show()
    
    # 生成比较报告 - 保持中文输出
    comparison_report = f"""# 各数据集评估比较报告

## 整体性能
| 数据集 | 准确率 | 平衡准确率 | 宏平均F1 | 加权F1 |
|--------|----------|--------------|----------|----------|
| 训练集 | {train_results['accuracy']:.4f} | {train_results['balanced_accuracy']:.4f} | {train_results['f1_macro']:.4f} | {train_results['f1_weighted']:.4f} |
| 测试集 | {test_results['accuracy']:.4f} | {test_results['balanced_accuracy']:.4f} | {test_results['f1_macro']:.4f} | {test_results['f1_weighted']:.4f} |
| 验证集 | {val_results['accuracy']:.4f} | {val_results['balanced_accuracy']:.4f} | {val_results['f1_macro']:.4f} | {val_results['f1_weighted']:.4f} |
| 合并集 | {merged_results['accuracy']:.4f} | {merged_results['balanced_accuracy']:.4f} | {merged_results['f1_macro']:.4f} | {merged_results['f1_weighted']:.4f} |

## 各大类准确率
| 大类 | 训练集 | 测试集 | 验证集 | 合并集 |
|------|--------|--------|--------|--------|
"""
    
    for i, name in enumerate(big_class_names):
        comparison_report += f"| {name} | {train_results['per_class_accuracy'][i]:.4f} | {test_results['per_class_accuracy'][i]:.4f} | {val_results['per_class_accuracy'][i]:.4f} | {merged_results['per_class_accuracy'][i]:.4f} |\n"
    
    comparison_report += f"""
## 结论
- 各数据集的整体准确率较为一致，说明模型具有良好的泛化能力。
- 各大类在不同数据集上的表现一致性较好，没有明显的过拟合现象。
- 模型在平衡准确率和加权F1得分方面表现良好，说明对各类别的识别相对平衡。

## 相关可视化
相关可视化图表已保存至: {results_dir}/all_datasets_comparison.png
"""
    
    # 保存比较报告
    with open(os.path.join(results_dir, 'datasets_comparison_report.md'), 'w') as f:
        f.write(comparison_report)
    
    print(f"\n评估完成! 结果已保存至: {results_dir}")
    return all_results

In [ ]:
# Main Execution Code 主执行代码
# 数据目录
DATA_DIRS = {
    'train_dir': "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/train",
    'test_dir': "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/test",
    'val_dir': "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/val"
}


"""主执行函数"""
print("开始脑体素分层分类系统训练...")

# 设置计算设备
device = torch.device(f"cuda:{DEVICE}" if DEVICE>=0 and torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# 1. 加载数据
print("\n1. 加载数据...")
from load_multiclass_data_from_dirs import load_multiclass_data_from_dirs

dataset_dict = load_multiclass_data_from_dirs(
    DATA_DIRS, 
    apply_pca=False,  # 不使用原始方法的PCA，我们将自己实现特征组PCA
    n_components=0, 
    norm=False  # 不使用原始方法的归一化，我们将自己实现标准化
)

# 提取数据
train_data = dataset_dict['train_samples']
train_labels = dataset_dict['train_labels']
test_data = dataset_dict['test_samples']
test_labels = dataset_dict['test_labels']
val_data = dataset_dict['val_samples']
val_labels = dataset_dict['val_labels']

print(f"训练集: {train_data.shape}, 标签: {train_labels.shape}")
print(f"测试集: {test_data.shape}, 标签: {test_labels.shape}")
print(f"验证集: {val_data.shape}, 标签: {val_labels.shape}")

# 2. 定义大类映射
print("\n2. 定义大类映射...")
fine_to_big, big_to_fine, big_class_names = define_big_classes()

# 将细分类标签映射为大类标签
train_big_labels = map_to_big_classes(train_labels, fine_to_big)
test_big_labels = map_to_big_classes(test_labels, fine_to_big)
val_big_labels = map_to_big_classes(val_labels, fine_to_big)

# 统计大类样本分布
print("\n大类样本分布:")
for i, name in enumerate(big_class_names):
    train_count = np.sum(train_big_labels == i)
    test_count = np.sum(test_big_labels == i)
    val_count = np.sum(val_big_labels == i)
    print(f"大类 {i} - {name}: 训练集 {train_count}, 测试集 {test_count}, 验证集 {val_count}")

# 3. 特征分组预处理
print("\n3. 特征分组与预处理...")
train_groups, pca_models = preprocess_all_feature_groups(train_data, apply_pca=APPLY_PCA, normalize=NORM)
# 使用相同的PCA和标准化处理测试集和验证集
test_groups = {}
val_groups = {}

for group_name, group_indices in [('diffusion', DIFF_FEATURES), ('qti', QTI_FEATURES), ('cest', CEST_FEATURES)]:
    # 提取特征
    test_group_data = test_data[:, group_indices]
    val_group_data = val_data[:, group_indices]
    
    # 标准化
    if NORM:
        scaler = StandardScaler()
        scaler.fit(train_data[:, group_indices])  # 使用训练集拟合
        test_group_data = scaler.transform(test_group_data)
        val_group_data = scaler.transform(val_group_data)
    
    # 应用PCA
    if APPLY_PCA and pca_models[group_name] is not None:
        test_group_data = pca_models[group_name].transform(test_group_data)
        val_group_data = pca_models[group_name].transform(val_group_data)
    
    test_groups[group_name] = test_group_data
    val_groups[group_name] = val_group_data

print("Determining optimal big classes...")
optimal_num_classes, fine_to_big, big_to_fine, big_class_names = determine_optimal_big_classes(
    train_data, train_labels, train_groups, min_classes=3, max_classes=7
)

# 4. 特征重要性分析
print("\n4. 特征重要性分析...")
importance_save_path = os.path.join(SAVE_PATH, "feature_importance.png")
importance_dict = analyze_feature_importance(train_groups, train_big_labels, importance_save_path)

# 5. 可视化类别分布
print("\n5. 可视化类别在特征空间的分布...")
visualize_feature_distributions(train_groups, train_big_labels, SAVE_PATH)

# 6. 创建数据加载器
print("\n6. 创建数据加载器...")
train_dataset = BrainVoxelDataset(train_groups, train_big_labels)
test_dataset = BrainVoxelDataset(test_groups, test_big_labels)
val_dataset = BrainVoxelDataset(val_groups, val_big_labels)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 7. 创建专家模型
print("\n7. 创建专家模型...")
diff_expert = ExpertKAN(train_groups['diffusion'].shape[1], DIFF_HIDDEN_DIM, optimal_num_classes, grid_size=FIXED_GRID).to(device)
qti_expert = ExpertKAN(train_groups['qti'].shape[1], QTI_HIDDEN_DIM, optimal_num_classes, grid_size=FIXED_GRID).to(device)
cest_expert = ExpertKAN(train_groups['cest'].shape[1], CEST_HIDDEN_DIM, optimal_num_classes, grid_size=FIXED_GRID).to(device)


expert_models = {
    'diffusion': diff_expert,
    'qti': qti_expert,
    'cest': cest_expert
}

# 8. 训练专家模型
print("\n8. 训练专家模型...")
expert_results = {}

for name, model in expert_models.items():
    print(f"\n开始训练 {name} 专家模型...")
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    
    result = train_expert_model(
        name, model, train_loader, val_loader, 
        criterion, optimizer, device, 
        num_epochs=EPOCH, val_epoch=VAL_EPOCH, 
        save_path=SAVE_PATH
    )
    
    expert_results[name] = result
    print(f"{name} 专家模型训练完成，最佳验证集准确率: {result['best_val_acc']:.4f} (Epoch {result['best_epoch']})")

# 9. 基于验证集性能设置专家权重
print("\n9. 设置专家权重...")
expert_weights = {}
for name, result in expert_results.items():
    expert_weights[name] = result['best_val_acc']

# 归一化权重
weight_sum = sum(expert_weights.values())
expert_weights = {name: weight/weight_sum for name, weight in expert_weights.items()}

print("专家权重:")
for name, weight in expert_weights.items():
    print(f"  {name}: {weight:.4f}")

# 10. 创建并训练集成模型
print("\n10. 创建并训练集成模型...")
ensemble_model = EnsembleExpertSystem(expert_models, expert_weights).to(device)
ensemble_criterion = nn.CrossEntropyLoss()
ensemble_optimizer = torch.optim.Adam(ensemble_model.parameters(), lr=LR/2, weight_decay=WEIGHT_DECAY)

ensemble_result = train_ensemble_model(
    ensemble_model, train_loader, val_loader, 
    ensemble_criterion, ensemble_optimizer, device, 
    num_epochs=EPOCH//2, val_epoch=VAL_EPOCH, 
    save_path=SAVE_PATH
)

print(f"集成模型训练完成，最佳验证集准确率: {ensemble_result['best_val_acc']:.4f} (Epoch {ensemble_result['best_epoch']})")

# 11. 评估最终模型
print("\n11. 评估最终模型...")
# 评估集成模型
print("评估集成模型...")
ensemble_eval = evaluate_model(ensemble_model, test_loader, device, is_ensemble=True)
visualize_evaluation_results(ensemble_eval, big_class_names, SAVE_PATH)

# 评估各专家模型
for name, model in expert_models.items():
    print(f"\n评估 {name} 专家模型...")
    # 创建单专家数据加载器
    test_single_groups = {name: test_groups[name]}
    test_single_dataset = BrainVoxelDataset(test_single_groups, test_big_labels)
    test_single_loader = DataLoader(test_single_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    expert_eval = evaluate_model(model, test_single_loader, device)
    expert_save_path = os.path.join(SAVE_PATH, name)
    visualize_evaluation_results(expert_eval, big_class_names, expert_save_path)

# 12. 生成总结报告
print("\n12. 生成总结报告...")
summary_report = f"""# 脑体素分层分类系统 - 训练总结报告

## 模型信息
- 模型名称: {MODEL_NAME}
- 数据集: {DATASET}
- 大类数量: {optimal_num_classes}
- 细分类别数量: {NUM_CLASS}
- 训练日期: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## 大类定义
"""

for i, name in enumerate(big_class_names):
    summary_report += f"- 大类 {i}: {name}\n"
    summary_report += f"  - 包含的细分类别: {big_to_fine[i]}\n"
    summary_report += f"  - 样本数量: 训练集 {np.sum(train_big_labels == i)}, 测试集 {np.sum(test_big_labels == i)}, 验证集 {np.sum(val_big_labels == i)}\n"

summary_report += f"""
## 特征分组
- 扩散特征 (1-15): {train_groups['diffusion'].shape[1]} 维 (原始 {len(DIFF_FEATURES)} 维)
- QTI特征 (16-225): {train_groups['qti'].shape[1]} 维 (原始 {len(QTI_FEATURES)} 维)
- CEST特征 (226-341): {train_groups['cest'].shape[1]} 维 (原始 {len(CEST_FEATURES)} 维)

## 专家模型性能
"""

for name, result in expert_results.items():
    summary_report += f"- {name} 专家:\n"
    summary_report += f"  - 最佳验证集准确率: {result['best_val_acc']:.4f} (Epoch {result['best_epoch']})\n"
    summary_report += f"  - 权重: {expert_weights[name]:.4f}\n"
    summary_report += f"  - 测试集准确率: {evaluate_model(result['model'], DataLoader(BrainVoxelDataset({name: test_groups[name]}, test_big_labels), batch_size=BATCH_SIZE), device)['accuracy']:.4f}\n"

summary_report += f"""
## 集成模型性能
- 最佳验证集准确率: {ensemble_result['best_val_acc']:.4f} (Epoch {ensemble_result['best_epoch']})
- 测试集准确率: {ensemble_eval['accuracy']:.4f}
- 平衡准确率: {ensemble_eval['balanced_accuracy']:.4f}
- 宏平均F1: {ensemble_eval['f1_macro']:.4f}
- 加权F1: {ensemble_eval['f1_weighted']:.4f}

## 大类性能分析
"""

for i, name in enumerate(big_class_names):
    summary_report += f"- 大类 {i} ({name}):\n"
    summary_report += f"  - 准确率: {ensemble_eval['per_class_accuracy'][i]:.4f}\n"

# 保存摘要报告
with open(os.path.join(SAVE_PATH, 'summary_report.md'), 'w') as f:
    f.write(summary_report)

# 在主执行代码的末尾，返回结果之前添加
print("\n13. 保存训练好的模型和数据...")
save_trained_models(
    ensemble_model=ensemble_model,
    expert_models=expert_models,
    pca_models=pca_models,
    fine_to_big=fine_to_big,
    big_to_fine=big_to_fine,
    big_class_names=big_class_names,
    optimal_num_classes=optimal_num_classes,
    save_path=SAVE_PATH
)
print(f"模型和相关数据已保存至: {SAVE_PATH}")

return ensemble_model, expert_models, pca_models, fine_to_big, big_to_fine, big_class_names



In [ ]:
def evaluate_all_datasets():
    """评估所有数据集（训练集、测试集、验证集、合并集）"""
    print("开始对所有数据集进行评估...")
    
    # 设置计算设备
    device = torch.device(f"cuda:{DEVICE}" if DEVICE>=0 and torch.cuda.is_available() else "cpu")
    
    # 加载训练好的模型
    model_path = os.path.join(SAVE_PATH, 'trained_models.pth')
    pca_path = os.path.join(SAVE_PATH, 'pca_models.pkl')
    
    ensemble_model, _, pca_models, fine_to_big, _, big_class_names = load_trained_models(
        model_path, pca_path, device
    )
    
    # 定义数据目录，包括合并目录
    data_dirs = {
        'train_dir': "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/train",
        'test_dir': "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/test",
        'val_dir': "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/val",
        'merged_dir': "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/merged"
    }
    
    # 评估所有数据集
    all_results = evaluate_on_all_datasets(
        ensemble_model, pca_models, fine_to_big, big_class_names, data_dirs, SAVE_PATH
    )
    
    print("所有数据集评估完成!")
    return all_results


    # 评估所有数据集
evaluate_all_datasets()

In [ ]:
# Inference and Evaluation Functions 推断和评估函数

def prepare_data_for_inference(data, pca_models, normalize=True):
    """
    准备推理数据，对数据进行相同的预处理
    
    参数:
        data: 输入数据
        pca_models: PCA模型字典
        normalize: 是否进行标准化
    
    返回:
        processed_groups: 处理后的特征组字典
    """
    processed_groups = {}
    
    # 处理扩散特征
    diff_data = data[:, DIFF_FEATURES]
    if normalize:
        scaler = StandardScaler()
        diff_data = scaler.fit_transform(diff_data)
    if pca_models['diffusion'] is not None:
        diff_data = pca_models['diffusion'].transform(diff_data)
    processed_groups['diffusion'] = diff_data
    
    # 处理QTI特征
    qti_data = data[:, QTI_FEATURES]
    if normalize:
        scaler = StandardScaler()
        qti_data = scaler.fit_transform(qti_data)
    if pca_models['qti'] is not None:
        qti_data = pca_models['qti'].transform(qti_data)
    processed_groups['qti'] = qti_data
    
    # 处理CEST特征
    cest_data = data[:, CEST_FEATURES]
    if normalize:
        scaler = StandardScaler()
        cest_data = scaler.fit_transform(cest_data)
    if pca_models['cest'] is not None:
        cest_data = pca_models['cest'].transform(cest_data)
    processed_groups['cest'] = cest_data
    
    return processed_groups

def predict_big_class(ensemble_model, data, pca_models, device):
    """
    使用集成模型预测大类
    
    参数:
        ensemble_model: 集成模型
        data: 输入数据
        pca_models: PCA模型字典
        device: 计算设备
    
    返回:
        predictions: 预测结果
        probabilities: 预测概率
    """
    # 准备数据
    processed_groups = prepare_data_for_inference(data, pca_models)
    
    # 转换为BrainVoxelDataset格式
    dataset = BrainVoxelDataset(processed_groups, np.zeros(len(data)), is_inference=True)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # 预测
    ensemble_model.eval()
    all_preds = []
    all_probs = []
    
    with torch.no_grad():
        for features in loader:
            # 将特征移动到设备
            features_on_device = {k: v.to(device) for k, v in features.items()}
            
            outputs, _ = ensemble_model(features_on_device)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    return np.array(all_preds), np.array(all_probs)

In [ ]:
# Model Export and Loading Functions 模型导出和加载函数
def save_trained_models(ensemble_model, expert_models, pca_models, fine_to_big, big_to_fine, big_class_names, optimal_num_classes, save_path):
    """
    保存训练好的模型和相关数据
    
    参数:
        ensemble_model: 集成模型
        expert_models: 专家模型字典
        pca_models: PCA模型字典
        fine_to_big: 细分类别到大类的映射
        big_to_fine: 大类到细分类别的映射
        big_class_names: 大类名称列表
        save_path: 保存路径
    """
    # 创建保存目录
    os.makedirs(save_path, exist_ok=True)
    
    # 保存模型状态
    model_dict = {
        'ensemble_state_dict': ensemble_model.state_dict(),
        'expert_weights': ensemble_model.expert_weights,
        'expert_state_dicts': {name: model.state_dict() for name, model in expert_models.items()},
        'fine_to_big': fine_to_big,
        'big_to_fine': big_to_fine,
        'big_class_names': big_class_names,
        'num_big_classes': optimal_num_classes  # 添加这一行
    }
    
    torch.save(model_dict, os.path.join(save_path, 'trained_models.pth'))
    
    # 保存PCA模型
    import pickle
    with open(os.path.join(save_path, 'pca_models.pkl'), 'wb') as f:
        pickle.dump(pca_models, f)
    
    print(f"模型和数据保存成功: {save_path}")

def load_trained_models(model_path, pca_path, device):
    """
    加载保存的模型和数据
    
    参数:
        model_path: 模型路径
        pca_path: PCA模型路径
        device: 计算设备
    
    返回:
        ensemble_model: 集成模型
        expert_models: 专家模型字典
        pca_models: PCA模型字典
        fine_to_big: 细分类别到大类的映射
        big_to_fine: 大类到细分类别的映射
        big_class_names: 大类名称列表
    """
    # 加载模型状态
    model_dict = torch.load(model_path, map_location=device)
    # 获取大类数量
    num_big_classes = model_dict.get('num_big_classes', DEFAULT_NUM_BIG_CLASSES)
    
    # 加载PCA模型
    import pickle
    with open(pca_path, 'rb') as f:
        pca_models = pickle.load(f)
    
    # 重建专家模型
    expert_models = {}
    for name, state_dict in model_dict['expert_state_dicts'].items():
        # 获取输入维度
        if name == 'diffusion':
            input_dim = pca_models[name].n_components_ if pca_models[name] else len(DIFF_FEATURES)
            hidden_dim = DIFF_HIDDEN_DIM
        elif name == 'qti':
            input_dim = pca_models[name].n_components_ if pca_models[name] else len(QTI_FEATURES)
            hidden_dim = QTI_HIDDEN_DIM
        else:  # 'cest'
            input_dim = pca_models[name].n_components_ if pca_models[name] else len(CEST_FEATURES)
            hidden_dim = CEST_HIDDEN_DIM
        
        model = ExpertKAN(input_dim, hidden_dim, num_big_classes, FIXED_GRID).to(device)
        model.load_state_dict(state_dict)
        expert_models[name] = model
    
    # 重建集成模型
    ensemble_model = EnsembleExpertSystem(expert_models, model_dict['expert_weights']).to(device)
    ensemble_model.load_state_dict(model_dict['ensemble_state_dict'])
    
    return (ensemble_model, expert_models, pca_models, 
            model_dict['fine_to_big'], model_dict['big_to_fine'], 
            model_dict['big_class_names'])

In [ ]:
# Complete System Testing 完整系统测试
def test_complete_system():
    """
    测试完整系统
    """
    print("测试完整分层分类系统...")
    
    # 设置计算设备
    device = torch.device(f"cuda:{DEVICE}" if DEVICE>=0 and torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")
    
    # 加载数据
    print("\n加载测试数据...")
    from load_multiclass_data_from_dirs import load_multiclass_data_from_dirs
    
    dataset_dict = load_multiclass_data_from_dirs(
        {'test_dir': DATA_DIRS['test_dir']}, 
        apply_pca=False,
        n_components=0, 
        norm=False
    )
    
    test_data = dataset_dict['test_samples']
    test_labels = dataset_dict['test_labels']
    
    print(f"测试数据: {test_data.shape}, 标签: {test_labels.shape}")
    
    # 加载训练好的模型
    print("\n加载训练好的模型...")
    model_path = os.path.join(SAVE_PATH, 'trained_models.pth')
    pca_path = os.path.join(SAVE_PATH, 'pca_models.pkl')
    
    ensemble_model, expert_models, pca_models, fine_to_big, big_to_fine, big_class_names = load_trained_models(
        model_path, pca_path, device
    )
    
    # 将细分类标签映射为大类标签
    test_big_labels = map_to_big_classes(test_labels, fine_to_big)
    
    # 准备数据
    print("\n准备测试数据...")
    processed_groups = prepare_data_for_inference(test_data, pca_models)
    
    # 创建数据集和加载器
    test_dataset = BrainVoxelDataset(processed_groups, test_big_labels)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # 评估集成模型
    print("\n评估集成模型...")
    ensemble_eval = evaluate_model(ensemble_model, test_loader, device, is_ensemble=True)
    visualize_evaluation_results(ensemble_eval, big_class_names, SAVE_PATH)
    
    print("\n测试完成!")
    return ensemble_eval

# 如果需要测试完整系统，取消下面的注释
# if __name__ == "__main__":
test_complete_system()